# format của data
1. Phải đưa về dạng dữ liệu của InstructABSA tức là có thể đưa về dạng text####[(a_ids, o_ids, sentiement)] -> nên đưa về dạng rỗng hết
2. Sau khi tách thì phải có cách nối với câu gốc -> sẽ tính metric dựa trên câu gốc này. 
3. Tính luôn cả span thì quá khó --> triplet (a,o,s) hoặc quadruplet (t,a,o,s) --> phải lưu lại giá trị predict entity type để xử lí và gán vào
4. format dự tính là một csv file trong đó có:
    - file ground truth : chứa text, bộ triplet, bộ quadruplet
    - file predict chính thức : sau khi đã qua các bước thêm topic và xử lí, gộp predict cho từng loại mô hình thì phải có cấu trúc giống file ground truth
    - file predicted ở level span sau khi dùng Instruct ABSA: phải có span, bộ triplet của span đó nếu được thì thêm cả index sentence để align
    - file prediction từ các mô hình SC: bắt buộc phải có span, topic (index sentence có thể xem xét sau)

In [1]:
import json
import pandas as pd
import ast

In [2]:
with open("/kaggle/working/test_id.json", "r") as f:
    data = json.load(f)
with open("/kaggle/working/formatted_prediction.json", "r") as f:
    preds = json.load(f)

In [3]:
# check data length
print(len(data))
print(len(preds))

926
926


In [4]:
# Nối, thêm index sentence
df = pd.read_csv("/kaggle/working/test.csv")

In [5]:
data[102]

{'index': 4112, 'sentence_id': 4697}

In [6]:
# điều kiện để nối là giống token, giống index sentence trong ids
df_ids = [i for i in range(len(df))]
for i,pred in enumerate(preds):
    flag = False
    pred_id_in_corpus = data[i]['index']
    pred_sentence_id = data[i]['sentence_id']
    pred_tokens = pred['tokens']
    pred_identifier = (pred_id_in_corpus, pred_sentence_id)
    for df_id in df_ids:
        df_sentence_id = df.iloc[df_id]['Index_sentence']
        text = " ".join(ast.literal_eval(df.iloc[df_id].label)['Token'])
        df_id_in_corpus = df.iloc[df_id]["Unnamed: 0"]
        df_identifier = (df_id_in_corpus, df_sentence_id)
        if pred_identifier == df_identifier:
            preds[i]['index_sentence'] = int(df_sentence_id)
            df_ids.remove(df_id)
            flag = True
            break
    if not flag:
        print(f"Không tìm thấy id phù hợp cho pred {i} với identifier {pred_identifier}")

In [8]:
# lưu lại file đã thêm
with open("/kaggle/working/formatted_prediction_SC.json", "w") as f:
    json.dump(preds, f, indent=4, ensure_ascii=False)

tách  thành các span, gắn thêm index_sentence và đổi sang định dạng phù hợp để đưa vào test InstructABSA

index_sentence,text,aspects,aspect_sentiment_pairs,aspect_opinion_pairs,triplets ->

index_sentence,text,[],"[]","[]","[]"

In [15]:
instructABSA_data = []
with open("/kaggle/working/formatted_prediction_SC.json", "r") as f:
    preds = json.load(f)

for pred in preds:
    if 'index_sentence' not in pred:
        print(f"Thiếu index_sentence trong pred: {pred}")
    else:
        index_sentence = pred['index_sentence']
        for p in pred['preds']:
            text = p['Span']
            aspect = []
            aspect_sentiment_pairs=str([])
            aspect_opinion_pairs = str([])
            triplets = str([])
            topic = p['Topic']
            instructABSA_data.append({
                "index_sentence": index_sentence,
                "text": text,
                "aspect": aspect,
                "aspect_sentiment_pairs": aspect_sentiment_pairs,
                "aspect_opinion_pairs": aspect_opinion_pairs,
                "triplets": triplets,
                "topic": topic
            })


In [26]:
# chuyển thành file csv
instructABSA_df = pd.DataFrame(instructABSA_data)
instructABSA_df.to_csv("/kaggle/working/instructABSA_test.csv", index=False)